In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# =========================================
# PATHS
# =========================================
DATA_ROOT = "BTMK"   # your dataset folder
NUM_CLASSES = 4
BATCH_SIZE = 32
EPOCHS = 10
LR = 3e-4

# =========================================
# TRANSFORMS
# =========================================
train_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

test_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

# =========================================
# LOAD DATA
# =========================================
train_data = datasets.ImageFolder(os.path.join(DATA_ROOT, "train"), transform=train_tf)
val_data   = datasets.ImageFolder(os.path.join(DATA_ROOT, "val"), transform=test_tf)
test_data  = datasets.ImageFolder(os.path.join(DATA_ROOT, "test"), transform=test_tf)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_data, batch_size=BATCH_SIZE)

print("Train:", len(train_data), " Val:", len(val_data), " Test:", len(test_data))
print("Classes:", train_data.classes)

from torchvision import models

# ============================
#   LOAD GOOGLENET CORRECTLY
# ============================
model = models.googlenet(
    weights=models.GoogLeNet_Weights.IMAGENET1K_V1,
    aux_logits=True
)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# ============================
#   TRAINING LOOP
# ============================
def run_epoch(loader, training=True):
    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0
    correct = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        if training:
            optimizer.zero_grad()

        out = model(x)

        # GoogLeNet returns tuple → extract main output
        if isinstance(out, tuple):
            out = out[0]

        loss = criterion(out, y)

        if training:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)


# =========================================
# TRAIN GOOGLENET
# =========================================
for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, True)
    val_loss, val_acc = run_epoch(val_loader, False)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

torch.save(model.state_dict(), "googlenet_finetuned.pth")
print("Saved fine-tuned GoogLeNet")

# ================================================================
# FEATURE EXTRACTION MODEL (REMOVE FC)
# ================================================================
class Identity(nn.Module):
    def forward(self, x):
        return x

feat_model = models.googlenet(weights=None, aux_logits=False)
feat_model.fc = Identity()
feat_model.load_state_dict(torch.load("googlenet_finetuned.pth"), strict=False)
feat_model = feat_model.to(device)
feat_model.eval()

def extract(loader):
    feats = []
    labels = []
    with torch.no_grad():
        for x,y in loader:
            x = x.to(device)
            f = feat_model(x)
            f = f.view(f.size(0), -1).cpu().numpy()
            feats.append(f)
            labels.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labels)

print("Extracting features...")
X_train, y_train = extract(train_loader)
X_test, y_test = extract(test_loader)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

# ================================================================
# TRAIN SVM
# ================================================================
svm = SVC(kernel="linear")
svm.fit(X_train_s, y_train)

pred = svm.predict(X_test_s)
print("SVM Test Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred, target_names=train_data.classes))

joblib.dump(svm, "svm_brain.joblib")
joblib.dump(scaler, "scaler_brain.joblib")

# ================================================================
# TRAIN KNN
# ================================================================
k = int(np.sqrt(len(X_train)))
knn = KNeighborsClassifier(n_neighbors=k)
knn.fit(X_train_s, y_train)

pred2 = knn.predict(X_test_s)
print("KNN Test Accuracy:", accuracy_score(y_test, pred2))
print(classification_report(y_test, pred2, target_names=train_data.classes))

joblib.dump(knn, "knn_brain.joblib")


DEVICE: cuda
Train: 4914  Val: 1052  Test: 1057
Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Epoch 1/10 | Train Acc: 0.8714 | Val Acc: 0.9468
Epoch 2/10 | Train Acc: 0.9571 | Val Acc: 0.9648
Epoch 3/10 | Train Acc: 0.9817 | Val Acc: 0.9810
Epoch 4/10 | Train Acc: 0.9849 | Val Acc: 0.9791
Epoch 5/10 | Train Acc: 0.9947 | Val Acc: 0.9829
Epoch 6/10 | Train Acc: 0.9925 | Val Acc: 0.9819
Epoch 7/10 | Train Acc: 0.9937 | Val Acc: 0.9848
Epoch 8/10 | Train Acc: 0.9959 | Val Acc: 0.9838
Epoch 9/10 | Train Acc: 0.9976 | Val Acc: 0.9819
Epoch 10/10 | Train Acc: 0.9957 | Val Acc: 0.9886
Saved fine-tuned GoogLeNet


c:\Users\sreeh\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\googlenet.py:47: FutureWarning: The default weight initialization of GoogleNet will be changed in future releases of torchvision. If you wish to keep the old behavior (which leads to long initialization times due to scipy/scipy#11299), please set init_weights=True.
  warnings.warn(


Extracting features...
SVM Test Accuracy: 0.8751182592242195
              precision    recall  f1-score   support

      glioma       0.80      0.83      0.81       244
  meningioma       0.78      0.79      0.78       248
     notumor       0.97      0.95      0.96       300
   pituitary       0.94      0.91      0.93       265

    accuracy                           0.88      1057
   macro avg       0.87      0.87      0.87      1057
weighted avg       0.88      0.88      0.88      1057

KNN Test Accuracy: 0.8240302743614002
              precision    recall  f1-score   support

      glioma       0.79      0.82      0.80       244
  meningioma       0.74      0.71      0.72       248
     notumor       0.95      0.83      0.88       300
   pituitary       0.81      0.94      0.87       265

    accuracy                           0.82      1057
   macro avg       0.82      0.82      0.82      1057
weighted avg       0.83      0.82      0.82      1057



['knn_brain.joblib']